# Project Overview

This notebook documents the development of a machine learning system for classifying patients into two categories based on the presence or absence of heart disease.

The project explores how supervised machine learning can be used to learn patterns from clinical features and classify new patients into a binary heart-disease outcome. The Cleveland dataset is used for model development, while the independent Hungary dataset is used as an external test set to assess how well the selected model generalizes beyond the development data.

The notebook follows a reproducible machine learning workflow, including data inspection, cleaning, missing-value handling, exploratory analysis, feature preprocessing, model comparison, hyperparameter tuning, stratified cross-validation, final model training, and evaluation on an independent dataset.

---

### Project Details

| **Item** | **Description** |
|---|---|
| **Project** | Cross-Dataset Heart Disease Classification |
| **Objective** | Develop and evaluate a binary heart-disease classification model |
| **Development Dataset** | Cleveland heart disease dataset |
| **External Test Dataset** | Hungary heart disease dataset |
| **Task** | Binary classification |
| **Target** | Heart disease presence or absence |
| **Categorical Features** | Sex, ChestPainType, FBS, RestECG, ExAng, Slope, Ca, Thal |
| **Numerical Features** | Age, RestBP, Chol, MaxHR, Oldpeak |
| **Models Evaluated** | Logistic Regression, Decision Tree |
| **Model Selection** | Stratified 5-fold cross-validation |
| **Evaluation Metric** | F1 score |
| **Final External Test** | Hungary dataset |

---

### Workflow

The project follows these stages:

1. Dataset loading and initial inspection
2. Data cleaning and validation
3. Exploratory data analysis
4. Feature and target preparation
5. Train-validation splitting
6. Missing-value imputation
7. Categorical encoding and numerical scaling
8. Model comparison and hyperparameter tuning
9. Stratified 5-fold cross-validation
10. Model selection
11. Final model training
12. External evaluation on the Hungary dataset
13. Model performance interpretation and generalization analysis

## Dataset and Problem Definition

The project uses two heart disease datasets from different patient populations:

* **Cleveland dataset** — used as the development dataset for training, validation, model selection, and cross-validation.
* **Hungary dataset** — reserved as an independent external test dataset for evaluating the selected model's generalization.

The datasets contain clinical and diagnostic measurements such as age, sex, resting blood pressure, cholesterol, maximum heart rate, exercise-induced angina, and other cardiovascular-related features.

### Prediction Target

The original target variable, `Num`, represents the presence and severity of heart disease using values from 0 to 4:

* `0` — no presence of heart disease
* `1–4` — presence of heart disease at different severity levels

For this project, the target is converted into a binary classification problem:

* `0` → **No heart disease**
* `1–4` → **Heart disease present**

This transformation allows the models to focus on the primary classification question: whether heart disease is present rather than predicting its severity.

### Development and External Evaluation Strategy

The Cleveland dataset is used to develop and select the models. It is divided into training and validation subsets, with stratified sampling used to preserve the target-class distribution.

After model selection, the training and validation data are combined to train the final model. The model is then evaluated once on the separate Hungary dataset, which was not used during model selection.

This separation provides an additional assessment of how the selected model performs on data from a different dataset rather than relying solely on performance within the Cleveland dataset.


In [ ]:
# Import the libraries and project-specific functions required throughout the notebook.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from matplotlib.colors import ListedColormap

from HW import *

# Set a fixed random seed to make random operations reproducible.
RSEED = 8

## 1. Dataset Loading and Initial Inspection

In [ ]:
# Load the Cleveland development dataset and Hungary external test dataset.
cleveland = pd.read_csv("cleveland.csv")
test = pd.read_csv("hungary.csv")

In [ ]:
# Inspect the first five rows of the Cleveland development dataset.
cleveland.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,53.0,1.0,3.0,130.0,246.0,1.0,2.0,173.0,0.0,0.0,1.0,3.0,3.0,0
1,54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0,3
2,222.0,1.0,4.0,125.0,249.0,1.0,2.0,144.0,1.0,1.2,2.0,1.0,3.0,1
3,58.0,1.0,4.0,100.0,234.0,0.0,0.0,156.0,0.0,0.1,1.0,1.0,7.0,2
4,51.0,0.0,4.0,130.0,305.0,0.0,0.0,142.0,1.0,1.2,2.0,0.0,7.0,2


In [ ]:
# Inspect the first five rows of the Hungary external test dataset.
test.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,47,0,2,140,257,0,0,135,0,1.0,1,?,?,0
1,52,1,4,112,342,0,1,96,1,1.0,2,?,?,1
2,41,0,2,125,184,0,-1,-1,0,0.0,?,?,?,0
3,58,1,4,135,222,0,0,100,0,0.0,?,?,?,0
4,54,0,2,140,309,?,1,140,0,0.0,?,?,?,0


In [ ]:
# Display descriptive statistics for the Cleveland development dataset.
cleveland.describe()

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000
mean,55.052805,0.693069,3.442244,131.623762,3543.082508,0.148515,1.277228,246.953795,0.343234,1.039604,1.597360,0.937294
std,16.236574,0.599270,5.080393,17.549467,57434.537877,0.356198,5.210380,1715.122158,0.540737,1.161075,0.622104,1.228536
min,-1.000000,-1.000000,1.000000,94.000000,-234.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000
25%,47.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,132.500000,0.000000,0.000000,1.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,240.000000,0.000000,1.000000,152.000000,0.000000,0.800000,2.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,274.500000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,2.000000
max,222.000000,7.000000,90.000000,200.000000,1000000.000000,1.000000,90.000000,30000.000000,5.000000,6.200000,3.000000,4.000000


In [ ]:
# Display descriptive statistics for the Hungary external test dataset.
test.describe()

,Age,Sex,ChestPainType,RestBP,Oldpeak,Num
count,293.000000,293.000000,293.000000,293.000000,293.000000,293.000000
mean,48.006826,0.733788,2.986348,132.583618,0.588055,0.361775
std,11.173903,0.486937,0.965049,17.607128,0.909554,0.481336
min,1.000000,0.000000,1.000000,92.000000,0.000000,0.000000
25%,42.000000,0.000000,2.000000,120.000000,0.000000,0.000000
50%,49.000000,1.000000,3.000000,130.000000,0.000000,0.000000
75%,54.000000,1.000000,4.000000,140.000000,1.000000,1.000000
max,170.000000,4.000000,4.000000,200.000000,5.000000,1.000000


## 2. Data Cleaning and Validation

In [ ]:
# Apply the data-cleaning procedure to both datasets and retain the resulting missing-value summaries.
cleveland_cleaned, missing_values_cleveland = clean_data(cleveland)
test_cleaned, missing_values_test = clean_data(test)

## 3. Feature Type Definition

The features are classified as categorical or numerical according to their data representation and clinical meaning. The target variable is excluded from both feature groups. This classification establishes the feature sets used for subsequent imputation and model preprocessing.

In [ ]:
# Define the categorical clinical features used for preprocessing.
categorical_columns = [
    "Sex",
    "ChestPainType",
    "FBS",
    "RestECG",
    "ExAng",
    "Slope",
    "Ca",
    "Thal"
]

# Define the numerical clinical features used for preprocessing.
numerical_columns = [
    "Age",
    "RestBP",
    "Chol",
    "MaxHR",
    "Oldpeak"
]

## 4. Train-Validation Splitting

The cleaned Cleveland dataset is divided into training and validation subsets using a stratified 70:30 split. Stratification preserves the distribution of the target classes across both subsets, while the indexes are reset to provide clean, sequential indexing for subsequent processing.

In [ ]:
# Separate the target variable from the input features in the Cleveland development dataset.
X_cleveland = cleveland_cleaned.drop(columns=["Num"])
y_cleveland = cleveland_cleaned["Num"]

# Separate the target variable from the input features in the Hungary external test dataset.
X_test = test_cleaned.drop(columns=["Num"])
y_test = test_cleaned["Num"]

# Create a stratified 70:30 training-validation split from the Cleveland dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_cleveland,
    y_cleveland,
    test_size=0.3,
    shuffle=True,
    stratify=y_cleveland,
    random_state=8
)

# Reset the indexes of the training and validation subsets after splitting.
X_train.reset_index(drop=True, inplace=True)
X_val.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_val.reset_index(drop=True, inplace=True)

In [ ]:
# Inspect the first five observations in the training feature set.
X_train.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
0,42.0,1.0,4.0,136.0,315.0,0.0,0.0,125.0,1.0,1.8,2.0,0.0,6.0
1,41.0,1.0,3.0,130.0,214.0,0.0,2.0,168.0,0.0,2.0,2.0,0.0,3.0
2,66.0,1.0,4.0,112.0,212.0,0.0,2.0,132.0,1.0,0.1,1.0,1.0,3.0
3,46.0,1.0,2.0,101.0,197.0,1.0,0.0,156.0,0.0,0.0,1.0,0.0,7.0
4,54.0,1.0,4.0,120.0,188.0,0.0,0.0,113.0,0.0,1.4,2.0,1.0,7.0


In [ ]:
# Inspect the first five observations in the validation feature set.
X_val.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
0,60.0,1.0,4.0,117.0,230.0,1.0,0.0,160.0,1.0,1.4,1.0,2.0,7.0
1,50.0,0.0,3.0,120.0,219.0,0.0,0.0,158.0,0.0,1.6,2.0,0.0,3.0
2,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0
3,53.0,1.0,4.0,142.0,226.0,0.0,2.0,111.0,1.0,0.0,1.0,0.0,7.0
4,43.0,1.0,4.0,132.0,247.0,1.0,2.0,143.0,1.0,0.1,2.0,NaN,7.0


In [ ]:
# Convert the original multi-level heart disease target into a binary outcome,
# where 0 represents no heart disease and values greater than 0 represent disease presence.
y_train = (y_train > 0).astype(int)
y_val = (y_val > 0).astype(int)
y_test = (y_test > 0).astype(int)

## 5. Missing-Value Imputation

### Categorical Feature Imputation

Missing values in the categorical features are imputed using the `impute_missing_categorical` procedure defined in `HW.py`. The imputation is fitted using the training data and applied to the training, validation, and external test feature sets.

In [ ]:
# Apply KNN-based imputation to the categorical features using the training data as the reference.
X_train_imputed_cat, X_val_imputed_cat, X_test_imputed_cat = impute_missing_categorical(
    X_train,
    X_val,
    X_test,
    categorical_columns
)

### Numerical Feature Imputation

Missing values in the numerical features are imputed using the `impute_numerical_features` procedure defined in `HW.py`. The imputation models are derived from the training data and applied to the training, validation, and external test feature sets.

In [ ]:
# Apply Lasso-based imputation to the numerical features using the training data as the reference.
X_train_numeric, X_val_numeric, X_test_numeric = impute_numerical_features(
    X_train,
    X_val,
    X_test,
    numerical_columns
)

### Merging Imputed Features

The imputed categorical and numerical feature sets are combined to reconstruct the complete feature matrices for the training, validation, and external test datasets.

In [ ]:
# Combine the imputed categorical and numerical features for the training dataset.
X_train_imputed = merge_imputed(
    X_train_imputed_cat,
    X_train_numeric
)

# Combine the imputed categorical and numerical features for the validation dataset.
X_val_imputed = merge_imputed(
    X_val_imputed_cat,
    X_val_numeric
)

# Combine the imputed categorical and numerical features for the external test dataset.
X_test_imputed = merge_imputed(
    X_test_imputed_cat,
    X_test_numeric
)

In [ ]:
X_train_imputed.head()

,Sex,ChestPainType,FBS,RestECG,ExAng,Slope,Ca,Thal,Age,RestBP,Chol,MaxHR,Oldpeak
0,1,4,0,0,1,2,0,6,42.0,136.0,315.0,125.0,1.8
1,1,3,0,2,0,2,0,3,41.0,130.0,214.0,168.0,2.0
2,1,4,0,2,1,1,1,3,66.0,112.0,212.0,132.0,0.1
3,1,2,1,0,0,1,0,7,46.0,101.0,197.0,156.0,0.0
4,1,4,0,0,0,2,1,7,54.0,120.0,188.0,113.0,1.4


### Imputed Feature Set Validation

In [ ]:
# Verify the dimensions of the imputed feature sets after combining categorical and numerical features.
print("Training shape:", X_train_imputed.shape)
print("Validation shape:", X_val_imputed.shape)
print("External test shape:", X_test_imputed.shape)

Training shape: (212, 13)
Validation shape: (91, 13)
External test shape: (293, 13)


## 6. Classification Using a Single Train-Validation Split

### Model Evaluation Using a Single Split

The classification models are evaluated using the training and validation subsets of the Cleveland development dataset. The `train_and_evaluate_single_split` function applies the specified preprocessing and hyperparameter configuration, trains the model on the training data, and evaluates its F1 score on the validation data.

In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Define the hyperparameter search space for the Decision Tree classifier.
hyperparameters_tree = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10]
}

# Define the hyperparameter search space for the Logistic Regression classifier.
hyperparameters_logreg = {
    "penalty": ["l1", "l2"],
    "C": [0.1, 10],
    "solver": ["liblinear"]
}

# Initialize a DataFrame to store the hyperparameter configurations and
# validation F1 scores from the single train-validation split.
performance_df = pd.DataFrame(columns=["params", "F1 scores"])

# Generate all possible hyperparameter combinations for each classifier.
tree_params = list(ParameterGrid(hyperparameters_tree))
logreg_params = list(ParameterGrid(hyperparameters_logreg))

# Report the number of configurations evaluated for each classifier.
print("Decision Tree configurations:", len(tree_params))
print("Logistic Regression configurations:", len(logreg_params))
print("Total configurations:", len(tree_params) + len(logreg_params))

# Evaluate each Decision Tree configuration using the Cleveland
# training and validation datasets.
for hp in tree_params:
    result = train_and_evaluate_single_split(
        X_train_imputed,
        X_val_imputed,
        y_train,
        y_val,
        categorical_columns,
        numerical_columns,
        DecisionTreeClassifier(random_state=RSEED),
        hp
    )

    performance_df = pd.concat(
        [performance_df, pd.DataFrame([result])],
        ignore_index=True
    )

# Evaluate each Logistic Regression configuration using the Cleveland
# training and validation datasets.
for hp in logreg_params:
    result = train_and_evaluate_single_split(
        X_train_imputed,
        X_val_imputed,
        y_train,
        y_val,
        categorical_columns,
        numerical_columns,
        LogisticRegression(random_state=RSEED),
        hp
    )

    performance_df = pd.concat(
        [performance_df, pd.DataFrame([result])],
        ignore_index=True
    )

# Rank all evaluated configurations by validation F1 score.
performance_df.sort_values(
    by="F1 scores",
    ascending=False
).reset_index(drop=True)

Decision Tree configurations: 6
Logistic Regression configurations: 4
Total configurations: 10


/tmp/ipykernel_38642/2100013957.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  performance_df = pd.concat(


,params,F1 scores
0,"{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}",0.837209
1,"{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}",0.827586
2,"{'C': 0.1, 'penalty': 'l1', 'solver': 'libline...",0.814815
3,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline...",0.813953
4,"{'criterion': 'gini', 'max_depth': 5}",0.714286
5,"{'criterion': 'entropy', 'max_depth': 3}",0.697674
6,"{'criterion': 'gini', 'max_depth': 10}",0.688889
7,"{'criterion': 'gini', 'max_depth': 3}",0.681818
8,"{'criterion': 'entropy', 'max_depth': 5}",0.681818
9,"{'criterion': 'entropy', 'max_depth': 10}",0.673913


### Final Development Dataset

In [ ]:
# Combine the Cleveland training and validation sets to create the final
# development dataset for model training after hyperparameter selection.
X = pd.concat(
    [X_train_imputed, X_val_imputed],
    ignore_index=True
)

y = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

# Verify the dimensions of the combined development dataset.
print("Combined Cleveland feature shape:", X.shape)
print("Combined Cleveland target shape:", y.shape)

Combined Cleveland feature shape: (303, 13)
Combined Cleveland target shape: (303,)


## 7. Classification Using Stratified Cross-Validation

This stage evaluates the Decision Tree and Logistic Regression models using stratified 5-fold cross-validation on the combined Cleveland development dataset. The same hyperparameter search spaces used in the single-split evaluation are retained, allowing model configurations to be compared using their average F1 score across the five validation folds.

In [ ]:
# Initialize a DataFrame to store the hyperparameter configurations
# and their average F1 scores from 5-fold cross-validation.
performance_df_cv = pd.DataFrame(
    columns=["params", "Average F1 scores"]
)

# Define the classifiers to be evaluated.
tree = DecisionTreeClassifier(random_state=RSEED)
logreg = LogisticRegression(random_state=RSEED)

# Generate all possible hyperparameter combinations for each classifier.
tree_grid = list(ParameterGrid(hyperparameters_tree))
logreg_grid = list(ParameterGrid(hyperparameters_logreg))

# Report the number of configurations evaluated for each classifier.
print("Decision Tree configurations:", len(tree_grid))
print("Logistic Regression configurations:", len(logreg_grid))
print("Total configurations:", len(tree_grid) + len(logreg_grid))

# Evaluate each Decision Tree configuration using stratified 5-fold cross-validation.
for hp in tree_grid:
    result = train_and_evaluate_cross_validation(
        X,
        y,
        tree,
        hp,
        cv=5,
        cat_cols=categorical_columns,
        num_cols=numerical_columns
    )

    performance_df_cv = pd.concat(
        [performance_df_cv, pd.DataFrame([result])],
        ignore_index=True
    )

# Evaluate each Logistic Regression configuration using stratified 5-fold cross-validation.
for hp in logreg_grid:
    result = train_and_evaluate_cross_validation(
        X,
        y,
        logreg,
        hp,
        cv=5,
        cat_cols=categorical_columns,
        num_cols=numerical_columns
    )

    performance_df_cv = pd.concat(
        [performance_df_cv, pd.DataFrame([result])],
        ignore_index=True
    )

# Rank all evaluated configurations by their average cross-validation F1 score.
performance_df_cv.sort_values(
    by="Average F1 scores",
    ascending=False
).reset_index(drop=True)

Decision Tree configurations: 6
Logistic Regression configurations: 4
Total configurations: 10


/tmp/ipykernel_38642/554238944.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  performance_df_cv = pd.concat(


,params,Average F1 scores
0,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline...",0.843447
1,"{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}",0.833676
2,"{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}",0.833432
3,"{'C': 0.1, 'penalty': 'l1', 'solver': 'libline...",0.829595
4,"{'criterion': 'gini', 'max_depth': 3}",0.801217
5,"{'criterion': 'entropy', 'max_depth': 3}",0.797560
6,"{'criterion': 'gini', 'max_depth': 10}",0.761918
7,"{'criterion': 'gini', 'max_depth': 5}",0.760451
8,"{'criterion': 'entropy', 'max_depth': 10}",0.749156
9,"{'criterion': 'entropy', 'max_depth': 5}",0.735146


In [ ]:
# Combine the imputed Cleveland training and validation features
# to form the final development feature set.
X = pd.concat(
    [X_train_imputed, X_val_imputed],
    ignore_index=True
)

# Combine the corresponding binary target values for the final development set.
y = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

# Verify the dimensions of the combined development dataset.
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (303, 13)
y shape: (303,)


### Development Dataset Validation

In [ ]:
# Verify that the combined development feature set contains no missing values
# before proceeding to model training.
print("Total missing values in development features:", X.isna().sum().sum())

Total missing values in development features: 0


### Final Model Configuration

In [ ]:
# Define the Logistic Regression model using the selected hyperparameters
# from the model evaluation and cross-validation stages.
final_model = LogisticRegression(
    C=0.1,
    penalty="l2",
    solver="liblinear",
    random_state=RSEED
)

In [ ]:
# Configure one-hot encoding for categorical features.
# Unknown categories encountered during external evaluation are ignored.
cat_transformer = OneHotEncoder(handle_unknown="ignore")

# Standardize numerical features so that they are on a comparable scale.
num_transformer = StandardScaler()

# Apply the appropriate preprocessing to each feature type.
preprocessor = ColumnTransformer([
    ("cat", cat_transformer, categorical_columns),
    ("num", num_transformer, numerical_columns)
])

# Combine preprocessing and the selected Logistic Regression model
# into a single pipeline for consistent training and prediction.
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_model)
])

In [ ]:
# Train the complete pipeline on the combined Cleveland development dataset.
final_pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Sex', 'ChestPainType',
                                                   'FBS', 'RestECG', 'ExAng',
                                                   'Slope', 'Ca', 'Thal']),
                                                 ('num', StandardScaler(),
                                                  ['Age', 'RestBP', 'Chol',
                                                   'MaxHR', 'Oldpeak'])])),
                ('model',
                 LogisticRegression(C=0.1, random_state=8,
                                    solver='liblinear'))])

In [ ]:
# Generate predictions for the independent Hungary dataset.
y_test_pred = final_pipeline.predict(X_test_imputed)

In [ ]:
# Calculate the F1 score on the independent Hungary dataset.
test_f1 = f1_score(y_test, y_test_pred)

# Report the model's external test performance.
print("Hungary Test F1 score:", test_f1)

Hungary Test F1 score: 0.7745098039215687


## 8. Interpretation of Results

### Model Selection

The 5-fold stratified cross-validation results identified Logistic Regression with `C=0.1`, `penalty='l2'`, and `solver='liblinear'` as the best-performing configuration. It achieved an average F1 score of 0.843 across the five cross-validation folds, outperforming the evaluated Decision Tree configurations.

The selected model was then retrained using the combined Cleveland training and validation datasets before being evaluated on the independent Hungary dataset.

### External Evaluation

The selected Logistic Regression model achieved an F1 score of 0.775 on the Hungary dataset. This is lower than the average cross-validation F1 score of 0.843 obtained on the Cleveland development data.

The difference between the development and external test performance indicates that the model's performance decreased when applied to an independent patient population. This indicates that the model's performance did not transfer fully from the Cleveland dataset to the independent Hungary dataset.

### Generalization Analysis

The difference in performance may be associated with differences between the Cleveland and Hungary datasets, including patient characteristics, feature distributions, and data collection practices. These differences can affect the patterns learned during model development and may reduce performance when the model is applied to a different population.

The external evaluation therefore provides a more realistic assessment of generalization than cross-validation on the Cleveland dataset alone. While the selected model demonstrates useful predictive performance, the lower Hungary F1 score highlights the importance of evaluating machine learning models on independent datasets before assuming that development-set performance will transfer to other populations.